# 43008 Reinforcement Learning -- Assignment 3, Part E: Value-Based Deep Reinforcement Learning on Snake

**The Deadly Triad (project 48): Yue Liu (25211451), Jiahao Liu (14191136), Siqin Jin (26145324)**

**Project**: a controlled study of state representation, reward design and DQN stabilisation on a self-built Snake
environment (8 x 8 and 10 x 10 boards): tabular Q-learning; DQN on an 11-feature vector (observation A, MLP) or an
8-plane grid (observation B8, CNN); the discount factor; the replay buffer and the target network; Double DQN, Dueling
and prioritised replay; four reward designs (R0 sparse, R1 step penalty, R2 potential-based shaping, R3 naive distance
bonus) and a reward-hacking probe; random, BFS-greedy and Hamiltonian-cycle reference players. Every configuration is
trained with five seeds, the seed is the unit of replication and every interval is a 95 % Student-t interval over seeds.

**What this notebook does**: it regenerates every table and figure of the report from the training logs.
`snake_rl.analysis.report` reads the run directories under `runs/` (per seed: `config.yaml`, `metrics.csv`,
`final_eval.json`), aggregates the seeds of every configuration and writes `runs/report/`: tables as Markdown, figures
as PNG, `index.md` and `manifest.json`. No number of the report is typed by hand; these files are the numbers.

**Training logs**: `runs/` is not in the repository (about 1 GB with checkpoints; the logs alone are about 12 MB,
4 MB zipped). Three ways to get them into this session: (1) upload `runs_logs.zip`, made with
`python scripts/export_logs.py` on the machine that trained, when the cell of Section 2 asks for it; (2) set `LOGS_URL`
in that cell to a link to the zip (for example a GitHub release asset); (3) run the notebook inside a checkout that
already has `runs/`.

**Behaviour analyses** (visit heat-maps, loop statistics, recordings of the longest loops) need the weights of a run,
which the zip does not carry; Section 5 runs one when a `model.pt` is present, and the composite heat-map figures of
Section 4.3 need the saved analyses (`python scripts/export_logs.py --include-behaviour`).

### 1. Repository and Imports

In [ ]:
# Clone the repository and install the package when running on Google Colab; a local checkout is used as it is
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Cureeeeeeee/Value-Based-Deep-Reinforcement-Learning-on-Snake.git"

if not os.path.isdir("snake_rl"):
    if not os.path.isdir("snake-rl"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "snake-rl"], check=True)
    os.chdir("snake-rl")
if IN_COLAB:
    %pip install -q -e .
print("working directory:", os.getcwd())

In [ ]:
import glob
import json
import urllib.request
import zipfile

import numpy as np
import torch
from IPython.display import Image, Markdown, display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"python {sys.version.split()[0]}  torch {torch.__version__}  numpy {np.__version__}  device {DEVICE}")

### 2. Training Logs

In [ ]:
# Training logs: the local runs/ directory, or runs_logs.zip (scripts/export_logs.py) from LOGS_URL or the Colab upload dialog
LOGS_URL = ""               # link to runs_logs.zip; empty: upload it on Colab, or use the local runs/
LOGS_ZIP = "runs_logs.zip"


def experiment_dirs():
    return sorted(d for d in glob.glob("runs/*/") if glob.glob(os.path.join(d, "*", "final_eval.json")))


if not experiment_dirs():
    if LOGS_URL:
        urllib.request.urlretrieve(LOGS_URL, LOGS_ZIP)
    elif IN_COLAB and not os.path.exists(LOGS_ZIP):
        from google.colab import files

        LOGS_ZIP = next(iter(files.upload()))
    elif not os.path.exists(LOGS_ZIP):
        raise FileNotFoundError("no runs/ and no runs_logs.zip: set LOGS_URL or copy the zip next to this notebook")
    with zipfile.ZipFile(LOGS_ZIP) as zf:
        zf.extractall(".")  # entries are runs/<experiment>/<seed>/<file>

print(len(experiment_dirs()), "experiment directories with finished runs under runs/")

### 3. Experiment Matrix

The 64 configurations of the study are defined in `snake_rl/analysis/matrix.py`; the status table lists the finished
seeds of every row and checks each run's recorded `config.yaml` against the row that launched it.

In [ ]:
# Finished seeds per configuration, and every run's config.yaml checked against the matrix
!{sys.executable} -m snake_rl.analysis.matrix --status

### 4. Tables and Figures of the Report

#### 4.1 Build from the logs

In [ ]:
# Every table and figure of the report; items whose experiments are missing are skipped and listed in runs/report/index.md
!{sys.executable} -m snake_rl.analysis.report --runs-dir runs --out-dir runs/report

#### 4.2 Display

One item of `manifest.json` per table and figure, with the experiments and seeds behind it; the `last-5` column of a
table is the mean of each agent's last five evaluations.

In [ ]:
# Show every item of the manifest: the Markdown table and the PNG figures
with open("runs/report/manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)
print("code", manifest["git"], "| last-k column:", manifest["last_k"], "|", len(manifest["items"]), "items")

for item in manifest["items"]:
    if not item["outputs"]:
        continue
    display(Markdown(f"##### {item['name']}\n\n{item.get('title') or ''}"))
    for kind, rel in item["outputs"].items():
        path = os.path.join("runs/report", rel)
        if kind == "table":
            with open(path, encoding="utf-8") as f:
                display(Markdown(f.read()))
        else:
            display(Image(filename=path))

#### 4.3 Composite figures (needs the saved behaviour analyses)

Figures 2, 9, 11 and 12 of the report: the score distributions and the death causes come from the run directories, the
heat-maps of several players on one colour scale from the saved behaviour analyses under `runs/behaviour/`.

In [ ]:
# Composite figures of the report; the heat-map panels need the behaviour analyses listed in scripts/report_extra_figures.py
sys.path.insert(0, "scripts")
from report_extra_figures import CAUSE_FIGURES, PANELS, SCORE_DISTRIBUTIONS

needed = sorted({name for _, _, panels in PANELS for _, name in panels})
missing = [name for name in needed if not os.path.isdir(os.path.join("runs/behaviour", name))]
if missing:
    print("behaviour analyses missing under runs/behaviour:", ", ".join(missing))
    print("export the logs with --include-behaviour, or run snake_rl.analysis.behaviour for these players first")
else:
    !{sys.executable} scripts/report_extra_figures.py --out-dir runs/report/figures
    for file_name in [p[0] for p in SCORE_DISTRIBUTIONS] + [c[0] for c in CAUSE_FIGURES] + [p[0] for p in PANELS]:
        display(Image(filename=os.path.join("runs/report/figures", file_name)))

### 5. Behaviour Analysis of One Run (needs weights)

`snake_rl.analysis.behaviour` plays 100 evaluation episodes with a trained network and writes the visit heat-map, the
loop / shaping statistics per episode and GUI-replayable recordings of the longest loops
(`python -m snake_rl.gui.app --replay <file>`). It needs a `model.pt`, which the logs zip does not carry: train a run
(`python -m snake_rl.train --config configs/dqn_B_8x8.yaml --seed 0 --set agent.gamma=0.99 --exp dqn_B8_W8_gamma099`)
or copy a run directory here.

In [ ]:
# Behaviour analysis of one run: 100 greedy episodes, heat-map, loop statistics and recordings of the longest loops
RUN_DIR = "runs/dqn_B8_W8_gamma099/0"   # any run directory with model.pt
OUT_DIR = "runs/behaviour/notebook"

if os.path.exists(os.path.join(RUN_DIR, "model.pt")):
    !{sys.executable} -m snake_rl.analysis.behaviour --player {RUN_DIR} --checkpoint final --device {DEVICE} --out-dir {OUT_DIR}
    display(Image(filename=os.path.join(OUT_DIR, "heatmap.png")))
    display(Image(filename=os.path.join(OUT_DIR, "last_steps.png")))
    with open(os.path.join(OUT_DIR, "summary.json"), encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2)[:2000])
else:
    print("no weights at", RUN_DIR, "- the logs zip carries none; see the note above")